In [1]:
import pandas as pd
import numpy as np

In [2]:
# define and check assumptions
np.random.seed(42)

data_scientist_data = {
    'group': ['Data Scientist'] * 20,
    'clicks': np.random.binomial(1,0.2, 20),
    'impressions': [10] * 20
}

ai_engineer = {
    'group': ['AI Engineer'] * 20,
    'clicks': np.random.binomial(1,0.8, 20),
    'impressions': [10] * 20
}

In [3]:
df_ds = pd.DataFrame(data_scientist_data)
df_ai = pd.DataFrame(ai_engineer)
df_combined = pd.concat([df_ds, df_ai], ignore_index=True)

df_combined.head()

,group,clicks,impressions
0,Data Scientist,0,10
1,Data Scientist,1,10
2,Data Scientist,0,10
3,Data Scientist,0,10
4,Data Scientist,0,10


In [4]:
df_combined.shape

(40, 3)

In [5]:
df_combined

,group,clicks,impressions
0,Data Scientist,0,10
1,Data Scientist,1,10
2,Data Scientist,0,10
3,Data Scientist,0,10
4,Data Scientist,0,10
5,Data Scientist,0,10
6,Data Scientist,0,10
7,Data Scientist,1,10
8,Data Scientist,0,10
9,Data Scientist,0,10


In [6]:
# Calcuate CTR
ctr_summary = df_combined.groupby('group').agg(
    total_clicks = ('clicks', 'sum'),
    total_impressions = ('impressions', 'sum')
)

ctr_summary['CTR'] = ctr_summary['total_clicks'] / ctr_summary['total_impressions']
print(ctr_summary)

                total_clicks  total_impressions    CTR
group                                                 
AI Engineer               17                200  0.085
Data Scientist             4                200  0.020


## Statistical Testing

In [7]:
from statsmodels.stats.proportion import proportions_ztest

In [8]:
successes = ctr_summary['total_clicks'].values   # number of clicks in each sample
nobs = ctr_summary['total_impressions'].values   # number of impressions in each sample

In [9]:
z_stat, p_value = proportions_ztest(successes, nobs, alternative='larger')

In [10]:
print(f"Z-Statistics: {z_stat:.4f}")
print(f"P-Value: {p_value:.4f}")

Z-Statistics: 2.9144
P-Value: 0.0018


## Confidence Interval

In [11]:
from statsmodels.stats.proportion import confint_proportions_2indep

In [12]:
# confidence interval
conf_int = confint_proportions_2indep(successes[0], nobs[0], successes[1], nobs[1], method='wald')

print(f"95% Confidence Interval for difference in CTR: {np.round(conf_int,6)}")

95% Confidence Interval for difference in CTR: [0.021753 0.108247]


## Stastical Power

In [13]:
from statsmodels.stats.power import GofChisquarePower

In [16]:
effect_size = (ctr_summary['CTR'].iloc[1] - ctr_summary['CTR'].iloc[0]) / np.sqrt(ctr_summary['CTR'].mean())

alpha = 0.05
power = 0.80

In [17]:
power_analysis = GofChisquarePower()
required_sample_size = power_analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power)

print(f"Required Sample Size for 80% Power: {required_sample_size:.0f} impressions per group")

Required Sample Size for 80% Power: 98 impressions per group


There are 80% chances of rejecting the Null Hyopothesis and 20% chances of accepting the Null Hypothesis